In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
import torch

In [2]:
unknown_token = "<unk>"
pad_token = "<pad>"
eos_token = "<eos>"

dataset_folder = "wikitext-103/"
save_folder = "save-data/"

tokenizer = Tokenizer(BPE(unk_token=unknown_token, dropout=0.1))
# We need a pre-tokenizer to speed up training - without one, the model has to study the entire dataset in one go!
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
trainer = BpeTrainer(special_tokens=[unknown_token, pad_token, eos_token], vocab_size=35000, min_frequency=2, initial_alphabet=ByteLevel.alphabet())

In [3]:
# Train the tokenizer and save it
train_files = [dataset_folder+"wiki.train.raw"]
tokenizer.train(train_files, trainer)
tokenizer_save_location = save_folder+"wikitext-tokenizer.json"
tokenizer.save(tokenizer_save_location)
print(f"Tokenizer trained and saved to {tokenizer_save_location}")

Tokenizer trained and saved to save-data/wikitext-tokenizer.json


In [4]:
# Algorithm to split up each Wikipedia article, so the model can distinguish between different articles
def preprocess_dataset(dataset_file):
    with open(dataset_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    articles = []
    current_article = []

    # First line is always a blank before the first article. Just remove it!
    for line in lines[1:]:
        clean_line = line.strip()
        # If this is the start of a new article, save the old article
        # Use a stripped version of the line to sidestep any whitespace shenanigans
        if clean_line.startswith("= ") and clean_line.endswith(" =") and len(clean_line) > 2 and clean_line[2] != '=' and current_article:
            articles.append("".join(current_article))
            current_article = []
        # Add to the current article
        current_article.append(line)
    
    # Add in the last article
    if current_article:
        articles.append("".join(current_article))

    # Add the eos token between each article, and at the end of the last article
    print(f"Detected {len(articles)} articles in {dataset_file}")
    preprocessed_dataset = eos_token.join(articles) + eos_token
    return preprocessed_dataset

In [ ]:
# Tokenize and save every dataset
dataset_files = [dataset_folder+"wiki.train.raw", dataset_folder+"wiki.test.raw", dataset_folder+"wiki.valid.raw", dataset_folder+"wiki.train.reduced.raw"]
token_files = [save_folder+"train-dataset-tokens.pt", save_folder+"test-dataset-tokens.pt", save_folder+"valid-dataset-tokens.pt", save_folder+"train-reduced-dataset-tokens.pt"]

for dataset_file, token_file in zip(dataset_files, token_files):
    preprocessed_dataset = preprocess_dataset(dataset_file)
    tokenized_dataset = tokenizer.encode(preprocessed_dataset)
    torch.save(torch.tensor(tokenized_dataset.ids, dtype=torch.long), token_file)
    print(f"{dataset_file} saved to {token_file}\n")

Detected 62 articles in wikitext-103/wiki.test.raw
wikitext-103/wiki.test.raw saved to save-data/test-dataset-tokens.pt
Detected 60 articles in wikitext-103/wiki.valid.raw
wikitext-103/wiki.valid.raw saved to save-data/valid-dataset-tokens.pt
Detected 205 articles in wikitext-103/wiki.train.reduced.raw
wikitext-103/wiki.train.reduced.raw saved to save-data/train-reduced-dataset-tokens.pt
